# 词向量运算

欢迎开始深度学习专项课程第 5 门课程、第 2 周的第一份作业！

因为词嵌入训练的计算代价非常高，大多数 ML 从业者会加载预训练一组嵌入。在这个 Notebook 你将尝试加载、测量之间的相似性以及修改预训练嵌入。 

**在此作业之后，你将能够**：

* 解释词嵌入如何捕获单词之间的关系
* 加载预训练词向量
* 使用余弦相似度测量词向量之间的相似度
* 使用词嵌入解决词语类比问题，例如：man 与 woman 的关系，类似于 king 与 ______ 的关系。

在本 Notebook 结束时，你将有机会尝试一项可选练习，你将修改词嵌入以减少性别偏见。减少偏见是 ML 中的一个重要考虑因素，因此我们鼓励你接受这一挑战！  

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/qHIve/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 加载词向量](#1)
- [2 - 嵌入向量与 one-hot 向量](#2)
- [3 - 余弦相似度](#3)
    - [练习 1 - cosine_similarity](#ex-1)
- [4 - 词语类比任务](#4)
    - [练习 2 - complete_analogy](#ex-2)
- [5 - 去偏词向量 (OPTIONAL/UNGRADED)](#5)
    - [5.1 - 消除非性别特定词语的偏见](#5-1)
        - [练习 3 - neutralize](#ex-3)
    - [5.2 - 性别特定词的均衡算法](#5-2)
        - [练习 4 - equalize](#ex-4)
- [6 - 参考文献](#6)

<a name='0'></a>
## 依赖库

让我们开始吧！运行以下单元格来加载你需要的包。

In [ ]:
### v1.1

In [ ]:
import numpy as np
from w2v_utils import *

<a name='1'></a>
## 1 - 加载词向量

对于此作业，你将使用 50 维 GloVe 向量来表示单词。 
运行以下单元格来加载 `word_to_vec_map`。 

In [ ]:
words, word_to_vec_map = read_glove_vecs('data/glove.6B.50d.txt')

你已加载：
- `words`：词汇表中的单词集。
- `word_to_vec_map`：将单词映射到其 GloVe 向量表示的字典。

<a name='2'></a>
## 2 - 嵌入向量与 one-hot 向量
回想一下课程视频，one-hot 向量不能很好地捕捉单词之间的相似程度。这是因为每个 one-hot 向量与任何其他 one-hot 向量具有相同的欧几里德距离。

嵌入向量，例如 GloVe 向量，提供了有关单个单词含义的更有用的信息。  
现在，看看如何使用 GloVe 向量来测量两个单词之间的相似度！ 

<a name='3'></a>
## 3 - 余弦相似度

要衡量两个单词之间的相似度，你需要一种方法来衡量这两个单词的两个嵌入向量之间的相似程度。给定两个向量 $u$ 和 $v$，余弦相似度定义如下： 

$$\text{CosineSimilarity(u, v)} = \frac {u \cdot v} {||u||_2 ||v||_2} = cos(\theta) \tag{1}$$

* $u \cdot v$ 是两个向量的点积（或内积）
* $||u||_2$ 是向量 $u$ 的范数（或长度）
* $\theta$ 是 $u$ 和 $v$ 之间的角度。 
* 余弦相似度取决于 $u$ 和 $v$ 之间的角度。 
    * 如果 $u$ 和 $v$ 非常相似，则它们的余弦相似度将接近 1。
    * 如果不相似，则余弦相似度取较小的值。 

<img src="images/cosine_sim.png" style="width:800px;height:250px;">
<div style="text-align:center;color:purple;"><b>图 1</b>：两个向量之间角度的余弦是它们的度量相似度.</div>

<a name='ex-1'></a>
### 练习 1 - cosine_similarity

实现函数 `cosine_similarity()` 来评估词向量之间的相似度。

**提醒**：$u$ 的范数定义为 $ ||u||_2 = \sqrt{\sum_{i=1}^{n} u_i^2}$

#### 附加提示
* 你可能会发现 [np.dot](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)、[np.sum](https://numpy.org/doc/stable/reference/generated/numpy.sum.html) 或 [np.sqrt](https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html) 很有用，具体取决于你选择的实现。

In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: cosine_similarity

def cosine_similarity(u, v):
    """
    余弦相似度反映了u和v的相似程度
        
    参数：
        u -- 形状为 (n,) 的词向量          
        v -- 形状为 (n,) 的词向量

    返回：
        cosine_similarity -- 由上式定义的 u 和 v 之间的余弦相似度。
    """
    
    # 特殊情况。考虑 u = [0, 0], v=[0, 0] 的情况
    if np.all(u == v):
        return 1
    
    ### START CODE HERE ###
    # 计算 u 和 v 之间的点积（≈1 行）
    dot = None 
    # 计算 u 的 L2 范数（≈1 行）
    norm_u = None
    
    # 计算 v 的 L2 范数（≈1 行）
    norm_v = None
    
    # 避免除以 0
    if np.isclose(norm_u * norm_v, 0, atol=1e-32):
        return 0
    
    # 计算公式（1）定义的余弦相似度（≈1 行代码）
    cosine_similarity = None
    ### END CODE HERE ###
    
    return cosine_similarity

In [ ]:
# 开始跳过评分
father = word_to_vec_map["father"]
mother = word_to_vec_map["mother"]
ball = word_to_vec_map["ball"]
crocodile = word_to_vec_map["crocodile"]
france = word_to_vec_map["france"]
italy = word_to_vec_map["italy"]
paris = word_to_vec_map["paris"]
rome = word_to_vec_map["rome"]

print("cosine_similarity(father, mother) = ", cosine_similarity(father, mother))
print("cosine_similarity(ball, crocodile) = ",cosine_similarity(ball, crocodile))
print("cosine_similarity(france - paris, rome - italy) = ",cosine_similarity(france - paris, rome - italy))
# 评分结束跳过

# 公开测试
def cosine_similarity_test(target):
    a = np.random.uniform(-10, 10, 10)
    b = np.random.uniform(-10, 10, 10)
    c = np.random.uniform(-1, 1, 23)
        
    assert np.isclose(cosine_similarity(a, a), 1), "cosine_similarity(a, a) must be 1"
    assert np.isclose(cosine_similarity((c >= 0) * 1, (c < 0) * 1), 0), "cosine_similarity(a, not(a)) must be 0"
    assert np.isclose(cosine_similarity(a, -a), -1), "cosine_similarity(a, -a) must be -1"
    assert np.isclose(cosine_similarity(a, b), cosine_similarity(a * 2, b * 4)), "cosine_similarity must be scale-independent. You must divide by the product of the norms of each input"

    print("\033[92mAll test passed!")
    
cosine_similarity_test(cosine_similarity)

#### 尝试不同的单词！

获得正确的预期输出后，请随意修改输入并测量其他单词对之间的余弦相似度！尝试其他输入的余弦相似性将使你更好地了解词向量的行为方式。

<a name='4'></a>
## 4 - 词语类比任务

* 在单词类比任务中，完成这句话：  
    <font color='brown'>"*a* 对应 *b* 正如 *c* 对应 **____**"</font>。 

* 一个例子是：  
    <font color='brown'> '*男人*对于*女人*就像*国王*对于*女王*'</font>。 

* 你正在尝试查找单词 *d*，使得关联的词向量 $e_a, e_b, e_c, e_d$ 按以下方式相关：   
    $e_b - e_a \approx e_d - e_c$
* 使用余弦相似度测量 $e_b - e_a$ 和 $e_d - e_c$ 之间的相似度。 

<a name='ex-2'></a>
### 练习 2 - complete_analogy

完成下面的代码来进行单词类比！

In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: complete_analogy

def complete_analogy(word_a, word_b, word_c, word_to_vec_map):
    """
    执行上述单词类比任务：a 对应 b，c 对应 ____。 
    
    参数：
    word_a -- 单词、字符串
    word_b -- 单词、字符串
    word_c -- 单词、字符串
    word_to_vec_map -- 将单词映射到相应向量的字典。 
    
    返回：
    best_word --  v_b - v_a 接近于 v_best_word - v_c 的单词，通过余弦相似度测量
    """
    
    # 将单词转换为小写
    word_a, word_b, word_c = word_a.lower(), word_b.lower(), word_c.lower()
    
    ### START CODE HERE ###
    # 获取词嵌入 e_a、e_b 和 e_c（约 1-3 行）
    e_a, e_b, e_c = None
    ### END CODE HERE ###
    
    words = word_to_vec_map.keys()
    max_cosine_sim = -100              # 初始化max_cosine_sim为一个大的负数
    best_word = None                   # 用 None 初始化 best_word，它将有助于跟踪要输出的单词
    
    # 循环整个词向量集合
    for w in words:   
        # 为避免 best_word 成为输入单词之一，请跳过输入 word_c
        # 从查询中跳过 word_c
        if w == word_c:
            continue
        
        ### START CODE HERE ###
        # 计算向量 (e_b - e_a) 和向量 ((w 的向量表示) - e_c) 之间的余弦相似度（≈1 行）
        cosine_sim = None
        
        # 如果目前看到的cosine_sim比max_cosine_sim多的话，
            # then：将新的max_cosine_sim设置为当前cosine_sim，将best_word设置为当前单词（约3行）
        if None > None:
            max_cosine_sim = None
            best_word = None
        ### END CODE HERE ###
        
    return best_word

In [ ]:
# 公开测试
def complete_analogy_test(target):
    a = [3, 3] # 中心位于
    a_nw = [2, 4] # 西北向向量
    a_s = [3, 2] # 来自 a 的南向向量
    
    c = [-2, 1] # 中心位于 c
    # 创建受控词到 vec 映射
    word_to_vec_map = {'a': a,
                       'synonym_of_a': a,
                       'a_nw': a_nw, 
                       'a_s': a_s, 
                       'c': c, 
                       'c_n': [-2, 2], # N
                       'c_ne': [-1, 2], # NE
                       'c_e': [-1, 1], # E
                       'c_se': [-1, 0], # SE
                       'c_s': [-2, 0], # S
                       'c_sw': [-3, 0], # SW
                       'c_w': [-3, 1], # W
                       'c_nw': [-3, 2] # 西北
                      }
    
    # 将列表转换为 np.arrays
    for key in word_to_vec_map.keys():
        word_to_vec_map[key] = np.array(word_to_vec_map[key])
            
    assert(target('a', 'a_nw', 'c', word_to_vec_map) == 'c_nw')
    assert(target('a', 'a_s', 'c', word_to_vec_map) == 'c_s')
    assert(target('a', 'synonym_of_a', 'c', word_to_vec_map) != 'c'), "Best word cannot be input query"
    assert(target('a', 'c', 'a', word_to_vec_map) == 'c')

    print("\033[92mAll tests passed")
    
complete_analogy_test(complete_analogy)

运行下面的单元格来测试你的代码。耐心点，小蚱蜢……这可能需要 1-2 分钟。

In [ ]:
# 开始跳过评分
triads_to_try = [('italy', 'italian', 'spain'), ('india', 'delhi', 'japan'), ('man', 'woman', 'boy'), ('small', 'smaller', 'large')]
for triad in triads_to_try:
    print ('{} -> {} :: {} -> {}'.format( *triad, complete_analogy(*triad, word_to_vec_map)))

# 评分结束跳过

获得输出后，尝试修改上面的输入单元格来测试你自己的类比。 

**提示**：尝试找到一些其他可行的类比对，以及其他一些算法无法给出正确答案的类比对：
    * 例如，你可以尝试小->小为大->？

## 恭喜！

你已完成作业的计分部分。到目前为止，你已经： 

* 加载了一些预训练词向量
* 使用余弦相似度测量词向量之间的相似度
* 使用词嵌入解决词语类比问题，例如：man 与 woman 的关系，类似于 king 与 __ 的关系。

余弦相似度是一种相对简单、直观但功能强大的方法，可用于捕获单词之间的细微关系。这些练习应该有助于你解释它是如何工作的，并将其应用到你自己的项目中！  

<div style="color:blue;">

<b>你应该记住什么</b>：

- 余弦相似度是比较词向量对之间相似度的好方法。
    - 请注意，L2（欧几里得）距离也适用。
- 对于 NLP 应用程序，使用预训练集词向量通常是一个很好的入门方式。 

</div>

即使你已经完成了计分部分，请查看此 Notebook 的其余部分，以了解如何对词向量进行去偏。

<a name='5'></a>
## 5 - 词向量去偏（可选/不计分） 

下面将考察词嵌入中反映的性别偏见，并探索减轻这种偏见的算法。除了了解去偏方法，本练习也有助于建立对词向量作用的直觉。本节涉及少量线性代数，但无需成为专家也能完成。这是一部分不计分的选做内容，欢迎尝试和探索。

首先，看看 GloVe 词嵌入与性别的关系。你将首先计算向量 $g = e_{woman}-e_{man}$，其中 $e_{woman}$ 表示与单词 *woman* 对应的词向量，而 $e_{man}$ 对应于与单词 *man* 对应的词向量。得到的向量 $g$ 粗略地编码了“性别”的概念。 

如果计算 $g_1 = e_{mother}-e_{father}$、$g_2 = e_{girl}-e_{boy}$ 等并对它们求平均值，你可能会得到更准确的表示，但目前仅使用 $e_{woman}-e_{man}$ 就可以得到足够好的结果。


In [ ]:
g = word_to_vec_map['woman'] - word_to_vec_map['man']
print(g)

现在，考虑不同单词与 $g$ 的余弦相似度。正的相似度值与负的余弦相似度意味着什么？ 

In [ ]:
print ('List of names and their similarities with constructed vector:')

# 女孩和男孩的名字
name_list = ['john', 'marie', 'sophie', 'ronaldo', 'priya', 'rahul', 'danielle', 'reza', 'katy', 'yasmin']

for w in name_list:
    print (w, cosine_similarity(word_to_vec_map[w], g))

如你所见，女性名字往往与我们构建的向量 $g$ 具有正余弦相似度，而男性名字往往具有负余弦相似度。这并不奇怪，而且结果似乎可以接受。 

现在尝试使用其他单词：

In [ ]:
print('Other words and their similarities:')
word_list = ['lipstick', 'guns', 'science', 'arts', 'literature', 'warrior','doctor', 'tree', 'receptionist', 
             'technology',  'fashion', 'teacher', 'engineer', 'pilot', 'computer', 'singer']
for w in word_list:
    print (w, cosine_similarity(word_to_vec_map[w], g))

这些结果是否出乎你的意料？它们反映了一些不恰当的性别刻板印象。例如，单词 "computer" 的投影为负，在数值上更接近男性名字；而 "literature" 的投影为正，更接近女性名字。

你将在下面看到如何使用 [Boliukbasi et al., 2016](https://arxiv.org/abs/1607.06520) 的算法来减少这些向量的偏见。请注意，某些单词对（例如“演员”/“女演员”或“祖母”/“祖父”）应保持特定性别，而其他单词（例如“接待员”或“技术”）应中和，即与性别无关。在消除偏见时，你必须以不同的方式对待这两种类型的单词。

<a name='5-1'></a>
### 5.1 - 消除非性别特定词语的偏见 

下图应该可以帮助你直观地了解中和的作用。如果你使用的是 50 维词嵌入，则 50 维空间可以分为两部分：偏见方向 $g$ 和其余 49 个维度，此处称为 $g_{\perp}$。在线性代数中，我们说 49 维 $g_{\perp}$ 与 $g$ 垂直（或“正交”），这意味着它与 $g$ 成 90 度。中和步骤采用 $e_{receptionist}$ 等向量，并将 $g$ 方向的分量清零，得到 $e_{receptionist}^{debiased}$。 

尽管 $g_{\perp}$ 是 49 维的，但考虑到在 2D 屏幕上可以绘制的内容的限制，它使用下面的一维轴进行说明。 

<img src="images/neutral.png" style="width:800px;height:300px;">
<div style="text-align:center;color:purple;"><b>图 2</b>：应用中和操作之前和之后表示的“接待员”词向量。</div>

<a name='ex-3'></a>
### 练习 3 - neutralize

实现 `neutralize()` 以消除“接待员”或“科学家”等词语的偏见。

给定一个输入嵌入 $e$，你可以使用以下公式来计算 $e^{debiased}$：

$$
e^{bias\_component} = \frac{e \cdot g}{||g||_2^2} * g\tag{2}
$$

$$
e^{debiased} = e - e^{bias\_component}\tag{3}
$$

如果你是线性代数专家，你可能会认为 $e^{bias\_component}$ 是 $e$ 在方向 $g$ 上的投影。如果你不是线性代数专家，请不要担心这一点。 ;)

**注意：** 除偏算法所来自的 [纸](https://papers.nips.cc/paper/6228-man-is-to-computer-programmer-as-woman-is-to-homemaker-debiasing-word-embeddings.pdf) 假设所有词向量的 L2 范数为 1，因此需要进行以下计算：

In [ ]:
# 本文假设所有词向量的 L2 范数为 1，因此需要进行此计算
from tqdm import tqdm
word_to_vec_map_unit_vectors = {
    word: embedding / np.linalg.norm(embedding)
    for word, embedding in tqdm(word_to_vec_map.items())
}
g_unit = word_to_vec_map_unit_vectors['woman'] - word_to_vec_map_unit_vectors['man']

In [ ]:
def neutralize(word, g, word_to_vec_map):
    """
    通过将 "word" 投影到与偏见轴正交的空间上来消除它的偏见。 
    这个函数确保性别中性词在性别子空间中为零。
    
    参数：
        word -- 指示要消除偏见的单词的字符串
        g -- NumPy-形状为(50,)的数组，对应偏见轴（比如性别）
        word_to_vec_map -- 字典将单词映射到其相应的向量。
    
    返回：
        e_debiased -- 输入 "word" 的中和词向量表示
    """
    
    ### START CODE HERE ###
    # 选择 "word" 的词向量表示形式。使用 word_to_vec_map。 （≈ 1 行）
    e = None
    
    # 使用上面给出的公式计算 e_biascomponent。 （≈ 1 行）
    e_biascomponent = None
 
    # 通过从中减去 e_biascomponent 来中和 e 
    # e_debiased 应等于其正交投影。 （≈ 1 行）
    e_debiased = None
    ### END CODE HERE ###
    
    return e_debiased

In [ ]:
word = "receptionist"
print("cosine similarity between " + word + " and g, before neutralizing: ", cosine_similarity(word_to_vec_map[word], g))

e_debiased = neutralize(word, g_unit, word_to_vec_map_unit_vectors)
print("cosine similarity between " + word + " and g_unit, after neutralizing: ", cosine_similarity(e_debiased, g_unit))

**预期输出**：第二个结果本质上是 0，直到数值舍入（按 $10^{-17}$ 的顺序）。


<table>
    <tr>
        <td>
            <b> 接待员和 g 之间的余弦相似度，在中和之前：</b> ：
        </td>
        <td>
         0.3307794175059374
        </td>
    </tr>
        <tr>
        <td>
            <b> 接待员与 g_unit 之间的余弦相似度，中和 </b> 后：
        </td>
        <td>
         3.5723165491646677e-17
        </td>
    </tr>
</table>

<a name='5-2'></a>
### 5.2 - 性别特定词的均衡算法

接下来，让我们看看如何将去偏应用于“演员”和“演员”等单词对。均衡适用于你可能希望仅通过性别属性进行区分的单词对。作为一个具体的例子，假设“女演员”比“演员”更接近“保姆”。通过对“保姆”应用中性化，你可以减少与保姆相关的性别刻板印象。但这仍然不能保证“演员”和“女演员”与“保姆”的距离相等。均衡算法可以解决这个问题。 

均衡背后的关键思想是确保特定的单词对与 49 维 $g_\perp$ 等距。均衡步骤还确保两个均衡步骤现在距 $e_{receptionist}^{debiased}$ 或距任何其他已被中和的功的距离相同。从视觉上看，这就是均衡的工作原理： 

<img src="images/equalize10.png" style="width:800px;height:400px;">


执行此操作的线性代数的推导有点复杂。 （有关详细信息，请参阅参考文献中的 Bolukbasi 等人，2016 年。）以下是关键方程：

$$
\mu = \frac{e_{w1} + e_{w2}}{2}\tag{4}
$$

$$
\mu_{B} = \frac {\mu \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} *\text{bias\_axis}
\tag{5}
$$

$$
\mu_{\perp} = \mu - \mu_{B} \tag{6}
$$

$$
e_{w1B} = \frac {e_{w1} \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} *\text{bias\_axis}
\tag{7}
$$

$$
e_{w2B} = \frac {e_{w2} \cdot \text{bias\_axis}}{||\text{bias\_axis}||_2^2} *\text{bias\_axis}
\tag{8}
$$

$$
e_{w1B}^{corrected} = \sqrt{{1 - ||\mu_{\perp} ||^2_2}} * \frac{e_{\text{w1B}} - \mu_B} {||e_{w1B} - \mu_B||_2} \tag{9}
$$

$$
e_{w2B}^{corrected} = \sqrt{{1 - ||\mu_{\perp} ||^2_2}} * \frac{e_{\text{w2B}} - \mu_B} {||e_{w2B} - \mu_B||_2} \tag{10}
$$

$$
e_1 = e_{w1B}^{corrected} + \mu_{\perp} \tag{11}
$$

$$
e_2 = e_{w2B}^{corrected} + \mu_{\perp} \tag{12}
$$

<a name='ex-4'></a>
### 练习 4 - equalize

实现下面的 `equalize()` 函数。 

使用上面的方程得到这对单词的最终均衡版本。祝你好运！

**提示**
- 使用[np.linalg.norm](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
def equalize(pair, bias_axis, word_to_vec_map):
    """
    按照上图描述的均衡方法对特定性别的单词进行去偏。
    
    参数：
    pair -- 待去偏的一对性别特定词，表示为字符串对，例如 ("actress", "actor")
    bias_axis -- 形状为 (50,) 的数组，对应于偏见轴 e.g 的向量。性别
    word_to_vec_map -- 将单词映射到其对应向量的字典
    
    返回
    e_1 -- 词向量对应第一个单词
    e_2 -- 词向量对应第二个单词
    """
    
    ### START CODE HERE ###
    # 步骤 1：选择 "word" 的词向量表示形式。使用 word_to_vec_map。 （≈ 2 行）
    w1, w2 = None
    e_w1, e_w2 = None
    
    # 步骤 2：计算 e_w1 和 e_w2 的平均值（≈ 1 行代码）
    mu = None

    # 步骤 3：计算 mu 在偏见轴和正交轴上的投影（≈ 2 行代码）
    mu_B = None
    mu_orth = None

    # 步骤4：使用方程（7）和（8）计算e_w1B和e_w2B（约2行）
    e_w1B = None
    e_w2B = None
        
    # 步骤5：使用上面给出的公式（9）和（10）调整e_w1B和e_w2B的Bias部分（约2行）
    corrected_e_w1B = None
    corrected_e_w2B = None

    # 第 6 步：将共同分量与各自校正后的投影相加，得到均衡处理后的 e1 和 e2，从而消除偏见（约 2 行代码）
    e1 = None
    e2 = None
                                                                
    ### END CODE HERE ###
    
    return e1, e2

In [ ]:
print("cosine similarities before equalizing:")
print("cosine_similarity(word_to_vec_map[\"man\"], gender) = ", cosine_similarity(word_to_vec_map["man"], g))
print("cosine_similarity(word_to_vec_map[\"woman\"], gender) = ", cosine_similarity(word_to_vec_map["woman"], g))
print()
e1, e2 = equalize(("man", "woman"), g_unit, word_to_vec_map_unit_vectors)
print("cosine similarities after equalizing:")
print("cosine_similarity(e1, gender) = ", cosine_similarity(e1, g_unit))
print("cosine_similarity(e2, gender) = ", cosine_similarity(e2, g_unit))

**预期输出**：

均衡前的余弦相似度：
<table>
    <tr>
        <td>
            <b>cosine_similarity(word_to_vec_map["男"],性别)</b> =
        </td>
        <td>
         -0.11711095765336832
        </td>
    </tr>
        <tr>
        <td>
            <b>cosine_similarity(word_to_vec_map["女人"], 性别)</b> =
        </td>
        <td>
         0.35666618846270376
        </td>
    </tr>
</table>

均衡后的余弦相似度：
<table>
    <tr>
        <td>
            <b>cosine_similarity（e1，性别）</b> =
        </td>
        <td>
         -0.23871136142883795
        </td>
    </tr>
        <tr>
        <td>
            <b>cosine_similarity（e2，性别）</b> =
        </td>
        <td>
         0.23871136142883792
        </td>
    </tr>
</table>

继续操作上面单元格中的输入单词，将均衡应用于其他单词对。 

提示：尝试...

这些去偏算法对于减少偏见非常有帮助，但并不完美，也不能消除所有偏见痕迹。例如，此实现的一个弱点是仅使用单词 _woman_ 和 _man_ 来定义偏见方向 $g$。如前所述，如果 $g$ 是通过计算 $g_1 = e_{woman} - e_{man}$ 来定义的； $g_2 = e_{mother} - e_{father}$； $g_3 = e_{girl} - e_{boy}$；依此类推并对它们进行平均，你将获得 50 维词嵌入空间中“性别”维度的更好估计。也可以随意尝试这些类型的变体！ 

### 恭喜！

你已完成本 Notebook 的计分与不计分部分，了解了词向量的多种应用和修改方法，并掌握了去偏中的中和与均衡操作。下次见！

<a name='6'></a>
## 6 - 参考文献

- 去偏算法来自 Bolukbasi 等人，2016 年，[男人之于计算机程序员，正如女人之于计算机程序员
家庭主妇？去偏词嵌入](https://papers.nips.cc/paper/6228-man-is-to-computer-programmer-as-woman-is-to-homemaker-debiasing-word-embeddings.pdf)
- GloVe 词嵌入的设计者为 Jeffrey Pennington、Richard Socher 和 Christopher D. Manning。 (https://nlp.stanford.edu/projects/glove/)
